In [16]:
import ollama
from loguru import logger

In [17]:
llm_model = "gpt-oss:20b"

#### Define Tool

In [18]:
def wikipedia(country: str) -> str:
    """"
    Wikipedia tool function that simulates fetching information about a country from Wikipedia.

    Args:
        country (str): The name of the country to query.
    
    Returns:
        str: A string containing the information retrieved from Wikipedia about the specified country.
    """
    try:
        capitals = {
            'Italy': 'Rome',
            'Spain': 'Madrid',
        }
        capital = capitals.get(country, "Capital not found.")
        print(f"Fetched from Wikipedia: The capital of {country} is {capital}.")
        return capital
    except Exception as e:
        logger.error(f"Error calling Wikipedia API: {e}")
        return "Error fetching data from Wikipedia."

### ReAct (Reason + Act)
ReAct is the most fundamental agentic design pattern. If you only learn one pattern, make it this one.

#### The idea
ReAct combines reasoning (thinking about what to do) with acting (actually doing it) in an interleaved loop. The agent:

* **Thinks** about the current situation
* **Acts** by calling a tool or taking a step
* **Observes** the result gathered

Repeats until the task is done

In [19]:
class ReActAgent:

    AVAILABLE_MEMORY_MODES = ["sliding_window"]

    def __init__(self, system="", temperature=0, max_iterations=3, memory_mode="sliding_window", window_size=2):

        if system and not isinstance(system, str):
            raise ValueError("System message must be a string.")
        if not isinstance(max_iterations, int) or max_iterations <= 0:
            raise ValueError("max_iterations must be a positive integer.")
        if memory_mode not in self.AVAILABLE_MEMORY_MODES:
            raise ValueError(f"Unknown memory_mode '{memory_mode}'. Choose from {self.AVAILABLE_MEMORY_MODES}")
        if memory_mode == "sliding_window":
            assert window_size > 0, "window_size must be a positive integer for sliding_window memory mode."

        self.system = system
        self.temperature = temperature
        self.sliding_memory = []
        self.tools_used = []
        self.max_iterations = max_iterations
        if self.system:
            self.system_message = {"role": "system", "content": self.system}
        self.sliding_memory = [self.system_message]
        self.window_size = window_size
        self.memory_mode = memory_mode

    def __call__(self, message):
        try:
            self.sliding_memory.append({"role": "user", "content": message})
            result = self.execute()
            self.sliding_memory.append({"role": "assistant", "content": result})
            return result
        except Exception as e:
            logger.error(f"Error during agent call: {e}")
            return "An error occurred while processing your request."

    def get_sliding_window_memory(self):
        self.sliding_memory = self.sliding_memory[-self.window_size+1:]
        if self.system_message not in self.sliding_memory:
            self.sliding_memory.insert(0, self.system_message)

    def execute(self):

        iterations = 0

        while iterations < self.max_iterations:

            iterations += 1
            logger.info(f"[ReAct] Iteration {iterations}/{self.max_iterations}")

            if len(self.sliding_memory) > self.window_size:
                if self.memory_mode == "sliding_window":
                    self.get_sliding_window_memory()
                    logger.debug(f"[ReAct] Memory after sliding window applied: {len(self.sliding_memory)} messages.")

            # --- THINK ---
            # The model reasons about what to do next
            response = ollama.chat(
                model=llm_model,
                messages=self.sliding_memory,
                options={"temperature": self.temperature},
                tools=[wikipedia]
            )

            message = response["message"]
            tool_calls = message.get("tool_calls", [])

            # --- ACT ---
            # The model decides to call tools or provide a final answer
            if tool_calls:

                self.sliding_memory.append({
                    "role": "assistant",
                    "content": message.get("content", ""),
                    "tool_calls": tool_calls
                })

                logger.info(f"[ReAct] Model decided to call {len(tool_calls)} tool(s).")

                for call in tool_calls:
                    tool_name = call.function.name
                    tool_args = call.function.arguments
                    logger.info(f"[ReAct] ACT  → calling '{tool_name}' with args {tool_args}")

                    # --- OBSERVE ---
                    # Run the tool and feed the result back
                    if tool_name == "wikipedia":
                        observation = wikipedia(tool_args["country"])
                    else:
                        observation = f"Unknown tool: {tool_name}"
                        logger.warning(f"[ReAct] Unknown tool requested: {tool_name}")

                    logger.info(f"[ReAct] OBSERVE → {str(observation)[:200]}...")
                    self.tools_used.append(tool_name)

                    self.sliding_memory.append({
                        "role": "tool",
                        "tool_name": tool_name,
                        "content": str(observation)
                    })

            else:
                # if no tool calls, model provides the final answer 
                final_answer = message.get("content", "")
                logger.info(f"[ReAct] Final answer reached after {iterations} iteration(s).")
                return final_answer

        # max iterations hit without a conclusive answer
        logger.warning(f"[ReAct] Max iterations ({self.max_iterations}) reached without final answer.")
        return "I was unable to reach a final answer within the allowed number of steps."

In [20]:
react_agent = ReActAgent(system="You are a helpful assistant that can use tools to answer questions.")
response = react_agent("Tell me about the weather of 3/5/2026 in Italy.")
print("Final response:", response)

2026-05-03 16:50:35.660 | INFO     | __main__:execute:49 - [ReAct] Iteration 1/3
2026-05-03 16:50:42.905 | INFO     | __main__:execute:78 - [ReAct] Model decided to call 1 tool(s).
2026-05-03 16:50:42.906 | INFO     | __main__:execute:83 - [ReAct] ACT  → calling 'wikipedia' with args {'country': 'Italy'}
2026-05-03 16:50:42.907 | INFO     | __main__:execute:93 - [ReAct] OBSERVE → Rome...
2026-05-03 16:50:42.908 | INFO     | __main__:execute:49 - [ReAct] Iteration 2/3
2026-05-03 16:50:42.908 | DEBUG    | __main__:execute:54 - [ReAct] Memory after sliding window applied: 2 messages.


Fetched from Wikipedia: The capital of Italy is Rome.


2026-05-03 16:50:45.049 | INFO     | __main__:execute:78 - [ReAct] Model decided to call 1 tool(s).
2026-05-03 16:50:45.050 | INFO     | __main__:execute:83 - [ReAct] ACT  → calling 'wikipedia' with args {'country': 'Rome'}
2026-05-03 16:50:45.051 | INFO     | __main__:execute:93 - [ReAct] OBSERVE → Capital not found....
2026-05-03 16:50:45.051 | INFO     | __main__:execute:49 - [ReAct] Iteration 3/3
2026-05-03 16:50:45.052 | DEBUG    | __main__:execute:54 - [ReAct] Memory after sliding window applied: 2 messages.


Fetched from Wikipedia: The capital of Rome is Capital not found..


2026-05-03 16:50:48.628 | INFO     | __main__:execute:105 - [ReAct] Final answer reached after 3 iteration(s).


Final response: Sure! How can I help you today? If you have a question about a country or anything else, just let me know.
